# Notebook 02: ML Model Training & Comparison

## Objectives
In this notebook, we will:
1. **Preprocess** the hyperspectral dataset (80/20 train/test split and Standard Scaling fitted strictly on $X_{train}$ to prevent **Data Leakage**).
2. Train four candidate regression models:
   - **Linear Regression** (Baseline)
   - **Support Vector Regression (SVR)**
   - **Random Forest Regressor**
   - **XGBoost Regressor**
3. Measure actual **Training Execution Time** for each algorithm.
4. Evaluate performance on unseen test data using **MAE**, **RMSE**, and **$R^2$ Score**.
5. Plot **Actual vs. Predicted Parity Plots** and **Hyperspectral Feature Importances**.
6. Save the **Best Performing Model** (`models/best_model.joblib`).

> **Dataset Label Notice**: Dataset source is `[DEMO DATA - BIO-OPTICAL SIMULATION]`. Performance metrics serve to demonstrate pipeline mechanics; replacement with real physical camera data is planned for production.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src directory to path
sys.path.append(os.path.abspath("../src"))
from preprocessing import prepare_data
from train import train_models
from evaluate import evaluate_models

sns.set_theme(style="whitegrid")
print("✓ Preprocessing, Training, and Evaluation modules loaded successfully!")

--- 
## Step 1: Preprocess Data & Perform Train/Test Split
- **Features ($X$)**: 51 continuous hyperspectral reflectance bands ($400nm - 900nm$).
- **Target ($y$)**: **Turbidity (NTU)**.
- **Train/Test Split**: 80% Training ($X_{train}, y_{train}$), 20% Testing ($X_{test}, y_{test}$).

In [ ]:
raw_data_path = os.path.join("..", "data", "raw", "water_quality_hyperspectral_data.csv")
X_train, X_test, y_train, y_test, band_cols, scaler = prepare_data(raw_data_path)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

--- 
## Step 2: Train Machine Learning Regressors
We train Linear Regression, SVR, Random Forest, and XGBoost, measuring exact execution time for each.

In [ ]:
models_dir = os.path.join("..", "models")
trained_models, training_times = train_models(X_train, y_train, models_dir=models_dir)

--- 
## Step 3: Evaluate & Compare Performance
We calculate **MAE**, **RMSE**, **$R^2$**, and export our summary comparison table.

In [ ]:
results_dir = os.path.join("..", "results")
df_comparison, best_model_name = evaluate_models(
    trained_models,
    training_times,
    X_test,
    y_test,
    band_cols,
    results_dir=results_dir,
    models_dir=models_dir
)

df_comparison

--- 
## Step 4: Display Visualizations
### 1. Parity Plot (Actual vs. Predicted)

In [ ]:
from PIL import Image
parity_img_path = os.path.join("..", "results", "graphs", "03_actual_vs_predicted.png")
if os.path.exists(parity_img_path):
    img = Image.open(parity_img_path)
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

### 2. Hyperspectral Feature Importance

In [ ]:
importance_img_path = os.path.join("..", "results", "graphs", "05_feature_importance.png")
if os.path.exists(importance_img_path):
    img = Image.open(importance_img_path)
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.show()